[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fiit-ba/zneus-2026/blob/main/labs/week_06/task_5_mnist_mlp.ipynb)

# Week 6 · MLP for classification: MNIST

Classify handwritten digits (0–9) from 28 × 28 grayscale images with a PyTorch MLP. Metric: **accuracy**, higher is better.

**Join Kaggle:** open the [Week 6 invitation link](https://www.kaggle.com/t/e3b24b434743440c88a614faad67367e), sign in, and accept the rules. Use your **AIS login** as your team name.

Last week your network produced one house value. This week it produces ten scores, one for each digit. The input is still a vector: the 784 pixels of an image. Your job is to turn those vectors into reliable class predictions.

Working in Colab? Follow the [Colab setup instructions](https://github.com/fiit-ba/zneus-2026/blob/main/labs/week_01_setup/README.md#google-colab) to open your private copy.

In [ ]:
# Colab setup (does nothing locally; W&B is optional)
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

SEED = 42
SUBSET = None  # e.g. 256 to debug by overfitting a tiny subset; None = all rows
# Works when Jupyter starts at the repository root or beside this notebook.
LAB_DIR = Path("labs/week_06") if Path("labs/week_06").is_dir() else Path(".")
DATA_DIR = Path(os.environ.get("ZNEUS_DATA_DIR", str(LAB_DIR / "data")))
KAGGLE_DIR = DATA_DIR / "kaggle"

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("device:", device)

## Data

Download and extract `train.csv`, `test.csv` and `sample_submission.csv` from the [competition's Data tab](https://www.kaggle.com/t/e3b24b434743440c88a614faad67367e) into `data/kaggle/` next to this notebook (or `$ZNEUS_DATA_DIR/kaggle/`). In Colab, upload the extracted files into `/content/data/kaggle/`.

From the repository root, you can download the ZIP with:

```bash
uv run kaggle competitions download -c zneus-2026-week-6-mnist -p labs/week_06/data/kaggle
```

Extract the downloaded ZIP into that same folder before running the next cell.

| File | Content |
|---|---|
| `train.csv` | 60 000 rows: `id`, `label`, `pixel0` … `pixel783` |
| `test.csv` | 10 000 rows: `id`, `pixel0` … `pixel783` |
| `sample_submission.csv` | format example: `id,label` |

Pixels are integers from 0 to 255 in row-major order. The target `label` is an integer from 0 to 9. IDs identify rows and must not be used as model inputs.

The loader supplies `X_all`, `y_all`, `X_test` and `test_ids`, keeping the original test order. It leaves pixel scaling to you. Without the competition files it uses torchvision's MNIST for **local practice only**, with different IDs and a different split.

In [ ]:
FEATURES = [f"pixel{i}" for i in range(784)]
train_path = KAGGLE_DIR / "train.csv"
test_path = KAGGLE_DIR / "test.csv"
if train_path.exists() != test_path.exists():
    raise FileNotFoundError(f"Put both train.csv and test.csv in {KAGGLE_DIR}.")
USING_KAGGLE = train_path.exists() and test_path.exists()

if USING_KAGGLE:
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    assert list(train_df.columns) == ["id", "label", *FEATURES], "Unexpected train.csv columns"
    assert list(test_df.columns) == ["id", *FEATURES], "Unexpected test.csv columns"
    assert train_df["id"].is_unique and test_df["id"].is_unique, "IDs must be unique"
    assert not train_df["id"].isin(test_df["id"]).any(), "Train and test IDs overlap"
    assert train_df["label"].isin(range(10)).all(), "Labels must be integers 0–9"
    X_all = train_df[FEATURES].to_numpy(dtype=np.float32)
    y_all = train_df["label"].to_numpy(dtype=np.int64)
    X_test = test_df[FEATURES].to_numpy(dtype=np.float32)
    test_ids = test_df["id"].to_numpy()
    y_test_local = None
    print("Loaded competition files from", KAGGLE_DIR)
else:
    from torchvision.datasets import MNIST

    local_train = MNIST(DATA_DIR, train=True, download=True)
    local_test = MNIST(DATA_DIR, train=False, download=True)
    X_all = local_train.data.numpy().reshape(-1, 784).astype(np.float32)
    y_all = local_train.targets.numpy()
    X_test = local_test.data.numpy().reshape(-1, 784).astype(np.float32)
    y_test_local = local_test.targets.numpy()
    test_ids = np.array([f"local-{i}" for i in range(len(X_test))])
    print("LOCAL PRACTICE: torchvision MNIST; these IDs cannot be submitted to Kaggle.")

for name, pixels in [("train", X_all), ("test", X_test)]:
    assert np.isfinite(pixels).all(), f"{name} contains NaN or inf"
    assert ((pixels >= 0) & (pixels <= 255)).all(), f"{name} pixels must be in [0, 255]"
if SUBSET is not None:
    assert isinstance(SUBSET, int) and 0 < SUBSET <= len(y_all)
    subset_idx = np.random.default_rng(SEED).choice(len(y_all), SUBSET, replace=False)
    X_all, y_all = X_all[subset_idx], y_all[subset_idx]
print(f"X_all {X_all.shape}   y_all {y_all.shape}   X_test {X_test.shape}")

## Your task

Train the best **fully connected MLP** you can. Do not use convolutions, pretrained models or external training data. Use only the supplied training labels; looking up test labels or matching test images to labelled copies of MNIST is not permitted.

Start by displaying some images (`X_all[i].reshape(28, 28)`) and checking the class counts. Reserve a validation set from `X_all` and `y_all` before training. Scale the pixel values, then build your data loaders and network. If you estimate any preprocessing statistics, fit them on the training portion only.

Your network should produce ten raw scores (logits) per image. With [`CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html), use integer class targets of dtype `torch.long` and pass logits directly; the loss handles the softmax calculation. For final predictions, choose the class with the largest score. Use evaluation mode and disable gradients during validation and prediction.

Compare training and validation loss and accuracy. Inspect a few misclassified images and explain one change you tried. The public Kaggle score uses only half of the test set, so use your own validation set to choose a model.

Your code below must end with **`test_pred`**: one integer digit per row of `X_test`, in the original order. If you predict in batches, keep the test loader unshuffled.

**Optional W&B:** use project `zneus-2026`, run names `week06-...`. No account or W&B link is required; offline logging is fine.

In [ ]:
# TODO: explore the images, split the labelled data, scale pixels, build and train
# an MLP, evaluate it, and predict X_test in its original order.
test_pred = ...

## Kaggle submission

The final cell checks the predictions and writes `submission.csv` beside the notebook. It must contain exactly `id,label`, one row per test ID. Local practice writes `submission_local.csv` instead.

Upload on Kaggle using **Submit Predictions**. Paste the commit link to your notebook in your private GitHub repository into the **submission description**. This is the hand-in channel. Add a W&B link only if you used it.

From the repository root:

```bash
uv run kaggle competitions submit -c zneus-2026-week-6-mnist -f labs/week_06/submission.csv -m "https://github.com/<your-github-username>/zneus-2026/commit/<sha>"
```

In [ ]:
assert test_pred is not ..., "Complete the solution cell: test_pred is still `...`."
if torch.is_tensor(test_pred):
    test_pred = test_pred.detach().cpu().numpy()
test_pred = np.asarray(test_pred)
assert test_pred.ndim == 1, "Supply a 1D array of class labels, not logits or probabilities."
assert len(test_pred) == len(test_ids), f"Expected {len(test_ids)} predictions, got {len(test_pred)}."
assert np.issubdtype(test_pred.dtype, np.number), "Predictions must be numeric."
assert np.isfinite(test_pred).all(), "Predictions contain NaN or inf."
assert np.isin(test_pred, np.arange(10)).all(), "Every prediction must be an integer from 0 to 9."

submission = pd.DataFrame({"id": test_ids, "label": test_pred.astype(np.int64)})
submission_path = LAB_DIR / ("submission.csv" if USING_KAGGLE else "submission_local.csv")
submission.to_csv(submission_path, index=False)
print(f"Wrote {submission_path}: {len(submission)} rows, columns {list(submission.columns)}")
if not USING_KAGGLE:
    print("Local practice accuracy:", float((test_pred == y_test_local).mean()))
    print("Do not upload this file to Kaggle. Download the competition files and rerun.")
submission.head()

## Before you hand in

**Deadline: 21 October 2026 at 23:59 (Europe/Bratislava, CEST).**

- [ ] The notebook runs top to bottom and is committed and pushed to your private repository. Leave `data/`, `wandb/` and submission files out of Git.
- [ ] Your instructors have accepted collaborator access and can open the commit link.
- [ ] `submission.csv` is on the Kaggle leaderboard under your AIS login.
- [ ] The submission description contains your commit link (and W&B link if used).

Individual work; 20 submissions per day; select one final submission. The public and private leaderboards each use 5 000 test images.